# Local sparse reconstruction (no GPU) -- unknown camera pose

Input: a plain folder of images, nothing else (no yaml poses, no segmentation masks).

Pipeline: `feature_extractor` -> `exhaustive_matcher` -> `mapper` (this is the step that
estimates camera pose **from scratch**, unlike the earlier `plant3_db` pipeline which took
Gazebo poses as known input and only ran `point_triangulator`).

Since there are no real segmentation masks for this dataset, we write plain full-frame
(all-white) masks -- this keeps every pixel, but preserves the exact `images/masks/sparse/0`
folder layout that the Colab dense-reconstruction script already expects, so nothing in that
script needs to change.

Output: `<OUTPUT_DIR>.zip`, ready to upload in Cell 2 of the Colab notebook.

In [1]:
import os
import re
import glob
import shutil
import subprocess
import zipfile

from PIL import Image

## Paths -- edit INPUT_DIR (and optionally OUTPUT_DIR) for your dataset

In [2]:
# ---- EDIT THESE ----
INPUT_DIR = os.path.expanduser(
    "/home/aturki/Desktop/JetCobot_internship_2026/Data_Captured/plant3_db"
)
OUTPUT_DIR = os.path.expanduser(
    "/home/aturki/Desktop/JetCobot_internship_2026/colmap_dataset/plant3_colmap_sparse"
)
# ---------------------

IMAGE_DIR   = os.path.join(OUTPUT_DIR, "images")
MASK_DIR    = os.path.join(OUTPUT_DIR, "masks")
SPARSE_ROOT = os.path.join(OUTPUT_DIR, "sparse")   # mapper writes 0/, 1/, ... under here
DB_PATH     = os.path.join(OUTPUT_DIR, "database.db")

CAMERA_MODEL  = "SIMPLE_RADIAL"  # unknown intrinsics -> let COLMAP self-calibrate focal + 1 distortion term.
                                   # Use "PINHOLE" instead only if you already trust a distortion-free lens model.
SINGLE_CAMERA = True              # all frames came from the same physical camera

In [3]:
os.makedirs(IMAGE_DIR, exist_ok=True)
os.makedirs(MASK_DIR, exist_ok=True)
os.makedirs(SPARSE_ROOT, exist_ok=True)

## Collect + copy images

In [4]:
IMG_EXTS = (".png", ".jpg", ".jpeg")

found = sorted(
    p for p in glob.glob(os.path.join(INPUT_DIR, "*"))
    if p.lower().endswith(IMG_EXTS)
)
print(f"Found {len(found)} images in {INPUT_DIR}")
assert found, "No images found -- check INPUT_DIR"

for src in found:
    shutil.copy2(src, os.path.join(IMAGE_DIR, os.path.basename(src)))

print(f"Copied {len(found)} images -> {IMAGE_DIR}")

Found 139 images in /home/aturki/Desktop/JetCobot_internship_2026/Data_Captured/plant3_db
Copied 139 images -> /home/aturki/Desktop/JetCobot_internship_2026/colmap_dataset/plant3_colmap_sparse/images


## Dummy full-frame masks

COLMAP's mask convention: mask file = `<image_filename>.png` (extension **appended**, not
replaced), stored under `masks/`. No real segmentation exists for this set, so we write a
plain white (255) mask at each image's resolution -- i.e. keep every pixel -- purely so the
folder layout matches what the unchanged Colab script expects.

In [5]:
for img_path in sorted(glob.glob(os.path.join(IMAGE_DIR, "*"))):
    fname = os.path.basename(img_path)
    with Image.open(img_path) as im:
        w, h = im.size
    mask = Image.new("L", (w, h), 255)
    mask.save(os.path.join(MASK_DIR, fname + ".png"))

print(f"Wrote {len(os.listdir(MASK_DIR))} full-frame masks -> {MASK_DIR}")

Wrote 139 full-frame masks -> /home/aturki/Desktop/JetCobot_internship_2026/colmap_dataset/plant3_colmap_sparse/masks


## 1/3 Feature extraction (CPU)

In [7]:
subprocess.run([
    "colmap", "feature_extractor",
    "--database_path", DB_PATH,
    "--image_path", IMAGE_DIR,
    "--ImageReader.single_camera", "1" if SINGLE_CAMERA else "0",
    "--ImageReader.camera_model", CAMERA_MODEL,
    "--SiftExtraction.use_gpu", "0",
], check=True)

W20260723 07:59:03.544431 10138 feature_extraction.cc:403] Your current options use the maximum number of threads on the machine to extract features. Extracting SIFT features on the CPU can consume a lot of RAM per thread for large images. Consider reducing the maximum image size and/or the first octave or manually limit the number of extraction threads. Ignore this warning, if your machine has sufficient memory for the current settings.
I20260723 07:59:03.545225 10139 misc.cc:198] 
Feature extraction
I20260723 07:59:04.565038 10156 feature_extraction.cc:254] Processed file [1/139]
I20260723 07:59:04.565119 10156 feature_extraction.cc:257]   Name:            frame_10.png
I20260723 07:59:04.565126 10156 feature_extraction.cc:283]   Dimensions:      640 x 480
I20260723 07:59:04.565133 10156 feature_extraction.cc:286]   Camera:          #1 - SIMPLE_RADIAL
I20260723 07:59:04.565142 10156 feature_extraction.cc:289]   Focal Length:    768.00px
I20260723 07:59:04.565160 10156 feature_extracti

CompletedProcess(args=['colmap', 'feature_extractor', '--database_path', '/home/aturki/Desktop/JetCobot_internship_2026/colmap_dataset/plant3_colmap_sparse/database.db', '--image_path', '/home/aturki/Desktop/JetCobot_internship_2026/colmap_dataset/plant3_colmap_sparse/images', '--ImageReader.single_camera', '1', '--ImageReader.camera_model', 'SIMPLE_RADIAL', '--SiftExtraction.use_gpu', '0'], returncode=0)

## 2/3 Matching (CPU)

Exhaustive matcher is the right default for an unordered set with unknown pose. If these
frames are actually an ordered trajectory (like the robot-arm sweeps), swap in
`sequential_matcher` for speed.

In [8]:
subprocess.run([
    "colmap", "exhaustive_matcher",
    "--database_path", DB_PATH,
    "--SiftMatching.use_gpu", "0",
], check=True)

I20260723 07:59:26.108156 10195 misc.cc:198] 
Exhaustive feature matching
I20260723 07:59:26.113353 10195 feature_matching.cc:231] Matching block [1/3, 1/3]
I20260723 08:00:50.988173 10195 feature_matching.cc:46]  in 84.875s
I20260723 08:00:50.992416 10195 feature_matching.cc:231] Matching block [1/3, 2/3]
I20260723 08:02:11.629241 10195 feature_matching.cc:46]  in 80.637s
I20260723 08:02:11.643769 10195 feature_matching.cc:231] Matching block [1/3, 3/3]
I20260723 08:03:04.857329 10195 feature_matching.cc:46]  in 53.214s
I20260723 08:03:04.862465 10195 feature_matching.cc:231] Matching block [2/3, 1/3]
I20260723 08:04:26.727497 10195 feature_matching.cc:46]  in 81.865s
I20260723 08:04:26.736395 10195 feature_matching.cc:231] Matching block [2/3, 2/3]
I20260723 08:05:40.099174 10195 feature_matching.cc:46]  in 73.363s
I20260723 08:05:40.127874 10195 feature_matching.cc:231] Matching block [2/3, 3/3]
I20260723 08:06:29.916347 10195 feature_matching.cc:46]  in 49.789s
I20260723 08:06:29.9

CompletedProcess(args=['colmap', 'exhaustive_matcher', '--database_path', '/home/aturki/Desktop/JetCobot_internship_2026/colmap_dataset/plant3_colmap_sparse/database.db', '--SiftMatching.use_gpu', '0'], returncode=0)

## 3/3 Mapper -- this is the step that estimates pose from scratch

In [9]:
subprocess.run([
    "colmap", "mapper",
    "--database_path", DB_PATH,
    "--image_path", IMAGE_DIR,
    "--output_path", SPARSE_ROOT,
], check=True)

models = sorted(
    d for d in os.listdir(SPARSE_ROOT)
    if os.path.isdir(os.path.join(SPARSE_ROOT, d))
)
print("Reconstructed sub-models:", models)
assert models, "mapper produced no reconstruction -- check matching/feature extraction above"

I20260723 08:09:57.482728 11567 misc.cc:198] 
Loading database
I20260723 08:09:57.485208 11567 database_cache.cc:54] Loading cameras...
I20260723 08:09:57.485340 11567 database_cache.cc:64]  1 in 0.000s
I20260723 08:09:57.485369 11567 database_cache.cc:72] Loading matches...
I20260723 08:09:57.573380 11567 database_cache.cc:78]  5778 in 0.088s
I20260723 08:09:57.573460 11567 database_cache.cc:94] Loading images...
I20260723 08:09:57.615814 11567 database_cache.cc:143]  139 in 0.042s (connected 139)
I20260723 08:09:57.616286 11567 database_cache.cc:154] Building correspondence graph...
I20260723 08:09:58.237102 11567 database_cache.cc:190]  in 0.621s (ignored 0)
I20260723 08:09:58.238044 11567 timer.cc:91] Elapsed time: 0.013 [minutes]
I20260723 08:09:58.244668 11567 misc.cc:198] 
Finding good initial image pair
I20260723 08:09:59.703274 11567 misc.cc:198] 
Initializing with image pair #14 and #139
I20260723 08:09:59.713447 11567 misc.cc:198] 
Global bundle adjustment


iter      cost      cost_change  |gradient|   |step|    tr_ratio  tr_radius  ls_iter  iter_time  total_time
   0  1.615195e+02    0.00e+00    2.16e+03   0.00e+00   0.00e+00  1.00e+04        0    1.79e-03    4.44e-03
   1  1.039677e+02    5.76e+01    2.50e+04   0.00e+00   6.86e-01  1.05e+04        1    5.91e-03    1.04e-02
   2  7.235961e+01    3.16e+01    3.67e+03   6.87e+00   1.00e+00  3.16e+04        1    5.90e-03    1.63e-02
   3  7.143296e+01    9.27e-01    1.96e+03   7.71e+00   9.34e-01  9.12e+04        1    5.23e-03    2.16e-02
   4  7.130182e+01    1.31e-01    9.81e+02   9.81e-01   4.72e-01  9.11e+04        1    5.31e-03    2.69e-02
   5  7.116619e+01    1.36e-01    7.66e+02   4.44e+00   5.28e-01  9.12e+04        1    4.01e-03    3.10e-02
   6  7.106983e+01    9.64e-02    7.76e+02   5.03e+00   4.19e-01  9.08e+04        1    3.54e-03    3.45e-02
   7  7.096590e+01    1.04e-01    7.84e+02   5.15e+00   4.25e-01  9.05e+04        1    4.23e-03    3.88e-02
   8  7.085958e+01    1.06e-

I20260723 08:10:00.026684 11567 misc.cc:205] 
Bundle adjustment report
------------------------
I20260723 08:10:00.026811 11567 bundle_adjustment.cc:942] 
    Residuals : 2720
   Parameters : 2047
   Iterations : 101
         Time : 0.30588 [s]
 Initial cost : 0.243685 [px]
   Final cost : 0.146978 [px]
  Termination : No convergence

I20260723 08:10:00.029410 11567 incremental_mapper.cc:160] => Filtered observations: 0
I20260723 08:10:00.029462 11567 incremental_mapper.cc:167] => Filtered images: 0
I20260723 08:10:00.042016 11567 misc.cc:198] 
Registering image #23 (3)
I20260723 08:10:00.042061 11567 incremental_mapper.cc:495] => Image sees 659 / 2427 points
I20260723 08:10:00.058068 11567 misc.cc:205] 
Pose refinement report
----------------------
I20260723 08:10:00.058122 11567 bundle_adjustment.cc:942] 
    Residuals : 1318
   Parameters : 6
   Iterations : 3
         Time : 0.00411081 [s]
 Initial cost : 0.107472 [px]
   Final cost : 0.106007 [px]
  Termination : Convergence

I202

iter      cost      cost_change  |gradient|   |step|    tr_ratio  tr_radius  ls_iter  iter_time  total_time
   0  7.080738e+01    0.00e+00    6.82e+01   0.00e+00   0.00e+00  1.00e+04        0    3.02e-03    6.23e-03
   1  6.817237e+01    2.64e+00    1.91e+01   0.00e+00   9.99e-01  3.00e+04        1    5.25e-03    1.15e-02
   2  6.817065e+01    1.72e-03    1.94e+00   4.37e-01   1.02e+00  9.00e+04        1    4.91e-03    1.64e-02
   3  6.816866e+01    1.99e-03    1.85e+01   1.42e+00   9.70e-01  2.70e+05        1    4.62e-03    2.11e-02
   4  6.816639e+01    2.27e-03    1.20e+02   3.68e+00   4.88e-01  2.70e+05        1    4.75e-03    2.59e-02
   5  6.816196e+01    4.42e-03    8.52e+01   3.11e+00   7.88e-01  3.34e+05        1    4.62e-03    3.05e-02
   6  6.815929e+01    2.67e-03    8.57e+01   3.12e+00   6.91e-01  3.54e+05        1    4.41e-03    3.50e-02
   7  6.815689e+01    2.40e-03    6.16e+01   2.65e+00   7.96e-01  4.46e+05        1    4.31e-03    3.93e-02
   8  6.815543e+01    1.46e-

I20260723 08:10:00.434381 11567 misc.cc:205] 
Pose refinement report
----------------------
I20260723 08:10:00.434454 11567 bundle_adjustment.cc:942] 
    Residuals : 1370
   Parameters : 6
   Iterations : 3
         Time : 0.00327992 [s]
 Initial cost : 0.0808792 [px]
   Final cost : 0.0805904 [px]
  Termination : Convergence

I20260723 08:10:00.443734 11567 incremental_mapper.cc:40] => Continued observations: 685
I20260723 08:10:00.461221 11567 incremental_mapper.cc:43] => Added observations: 115
I20260723 08:10:00.543980 11567 misc.cc:205] 
Bundle adjustment report
------------------------
I20260723 08:10:00.544036 11567 bundle_adjustment.cc:942] 
    Residuals : 5838
   Parameters : 2299
   Iterations : 9
         Time : 0.0779381 [s]
 Initial cost : 0.104019 [px]
   Final cost : 0.0966195 [px]
  Termination : Convergence

I20260723 08:10:00.564774 11567 incremental_mapper.cc:78] => Merged observations: 0
I20260723 08:10:00.564836 11567 incremental_mapper.cc:79] => Completed observ

...

I20260723 08:14:06.832242 11567 misc.cc:205] 
Bundle adjustment report
------------------------
I20260723 08:14:06.832329 11567 bundle_adjustment.cc:942] 
    Residuals : 455350
   Parameters : 64990
   Iterations : 7
         Time : 7.54451 [s]
 Initial cost : 0.302907 [px]
   Final cost : 0.301557 [px]
  Termination : Convergence

I20260723 08:14:07.186441 11567 incremental_mapper.cc:175] => Completed observations: 17
I20260723 08:14:07.309687 11567 incremental_mapper.cc:178] => Merged observations: 0
I20260723 08:14:07.342113 11567 incremental_mapper.cc:160] => Filtered observations: 0
I20260723 08:14:07.342172 11567 incremental_mapper.cc:119] => Changed observations: 0.000075
I20260723 08:14:07.342204 11567 incremental_mapper.cc:167] => Filtered images: 0
I20260723 08:14:07.350589 11567 misc.cc:198] 
Registering image #121 (136)
I20260723 08:14:07.350622 11567 incremental_mapper.cc:495] => Image sees 877 / 1389 points
I20260723 08:14:07.391330 11567 misc.cc:205] 
Pose refinement re

iter      cost      cost_change  |gradient|   |step|    tr_ratio  tr_radius  ls_iter  iter_time  total_time
   0  4.574165e+04    0.00e+00    7.36e+03   0.00e+00   0.00e+00  1.00e+04        0    2.05e-01    2.01e+00
   1  4.425924e+04    1.48e+03    3.93e+02   0.00e+00   1.01e+00  3.00e+04        1    2.62e+00    4.63e+00
   2  4.425211e+04    7.14e+00    6.43e+01   2.23e-01   9.84e-01  9.00e+04        1    8.22e-01    5.46e+00
   3  4.425203e+04    7.75e-02    4.62e+00   7.26e-02   9.19e-01  2.19e+05        1    8.34e-01    6.29e+00
   4  4.425202e+04    3.33e-03    1.98e+00   4.26e-02   9.28e-01  5.87e+05        1    8.33e-01    7.12e+00
   5  4.425202e+04    1.44e-04    7.31e-01   1.12e-02   9.46e-01  1.76e+06        1    8.07e-01    7.93e+00


I20260723 08:14:19.414034 11567 misc.cc:205] 
Bundle adjustment report
------------------------
I20260723 08:14:19.414134 11567 bundle_adjustment.cc:942] 
    Residuals : 464864
   Parameters : 66223
   Iterations : 6
         Time : 7.95493 [s]
 Initial cost : 0.313684 [px]
   Final cost : 0.308534 [px]
  Termination : Convergence

I20260723 08:14:19.771018 11567 incremental_mapper.cc:175] => Completed observations: 2
I20260723 08:14:19.891619 11567 incremental_mapper.cc:178] => Merged observations: 28
I20260723 08:14:19.926437 11567 incremental_mapper.cc:160] => Filtered observations: 47
I20260723 08:14:19.926590 11567 incremental_mapper.cc:119] => Changed observations: 0.000331
I20260723 08:14:19.926640 11567 incremental_mapper.cc:167] => Filtered images: 0


Reconstructed sub-models: ['0']


I20260723 08:14:19.995914 11567 timer.cc:91] Elapsed time: 4.375 [minutes]


## Pick the best sub-model and make sure it lives at `sparse/0`

With unknown pose, `mapper` can split the set into multiple disconnected sub-models if some
frames don't share enough overlap to register (this is expected and different from the
known-pose pipeline, where 201/201 frames always registered because poses were given, not
estimated). We keep only the largest sub-model, at `sparse/0`, since that's the path the
Colab script's zip layout expects.

In [10]:
def analyze_model(path):
    result = subprocess.run(
        ["colmap", "model_analyzer", "--path", path],
        check=True, capture_output=True, text=True,
    )
    log = result.stderr
    def grab(pattern, cast=int):
        m = re.search(pattern, log)
        return cast(m.group(1)) if m else None
    return {
        "path": path,
        "registered_images": grab(r"Registered images:\s*(\d+)"),
        "points": grab(r"Points:\s*(\d+)"),
        "mean_track_length": grab(r"Mean track length:\s*([\d.]+)", float),
        "mean_reproj_error_px": grab(r"Mean reprojection error:\s*([\d.]+)px", float),
    }

reports = [analyze_model(os.path.join(SPARSE_ROOT, m)) for m in models]
for r in reports:
    print(r)

best = max(reports, key=lambda r: r["registered_images"] or 0)
print("\nBest sub-model:", best)

best_dir = best["path"]
if os.path.basename(best_dir) != "0":
    zero_dir = os.path.join(SPARSE_ROOT, "0")
    if os.path.exists(zero_dir):
        shutil.rmtree(zero_dir + "_discarded", ignore_errors=True)
        shutil.move(zero_dir, zero_dir + "_discarded")
    shutil.move(best_dir, zero_dir)

for m in models:
    d = os.path.join(SPARSE_ROOT, m)
    if os.path.isdir(d) and os.path.basename(d) != "0":
        shutil.rmtree(d, ignore_errors=True)

print("\nFinal model kept at:", os.path.join(SPARSE_ROOT, "0"))

{'path': '/home/aturki/Desktop/JetCobot_internship_2026/colmap_dataset/plant3_colmap_sparse/sparse/0', 'registered_images': 139, 'points': 21787, 'mean_track_length': 10.665443, 'mean_reproj_error_px': 0.343984}

Best sub-model: {'path': '/home/aturki/Desktop/JetCobot_internship_2026/colmap_dataset/plant3_colmap_sparse/sparse/0', 'registered_images': 139, 'points': 21787, 'mean_track_length': 10.665443, 'mean_reproj_error_px': 0.343984}

Final model kept at: /home/aturki/Desktop/JetCobot_internship_2026/colmap_dataset/plant3_colmap_sparse/sparse/0


## Reprojection error check -- pass/fail gate before zipping

In [11]:
final = analyze_model(os.path.join(SPARSE_ROOT, "0"))
print(final)

if final["mean_reproj_error_px"] is not None and final["mean_reproj_error_px"] < 2.0:
    print("Reprojection error looks healthy -- ready to zip and send to Colab.")
else:
    print("[!] Reprojection error is high (or missing) -- inspect matching/feature extraction before proceeding.")

{'path': '/home/aturki/Desktop/JetCobot_internship_2026/colmap_dataset/plant3_colmap_sparse/sparse/0', 'registered_images': 139, 'points': 21787, 'mean_track_length': 10.665443, 'mean_reproj_error_px': 0.343984}
Reprojection error looks healthy -- ready to zip and send to Colab.


## Zip for Colab

Layout matches exactly what the Colab script's Cell 3 unzip check expects: `images/`,
`masks/`, `sparse/0`. The raw feature-matching database isn't needed on the Colab side, so
it's excluded.

In [12]:
zip_path = OUTPUT_DIR + ".zip"
if os.path.exists(zip_path):
    os.remove(zip_path)

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(OUTPUT_DIR):
        for f in files:
            full = os.path.join(root, f)
            if full == DB_PATH:
                continue
            zf.write(full, os.path.relpath(full, OUTPUT_DIR))

print(f"Wrote {zip_path}")
print("Upload this file when Cell 2 of the Colab notebook prompts for colmap_sparse.zip")

Wrote /home/aturki/Desktop/JetCobot_internship_2026/colmap_dataset/plant3_colmap_sparse.zip
Upload this file when Cell 2 of the Colab notebook prompts for colmap_sparse.zip
